We previously explored GLM via Lasso+Gaussian with log10, and achieve 0.07 R^2, We might wanna explore Beta regression with lasso penalty, which is more suitable for our data. We will use the same train/test split as before, and use the same set of covariates.

In [2]:
# load lib
library(tidyverse)
library(glmnet)
library(glmmTMB)
library(caret)
library(DHARMa)
library(readr)
library(dplyr)
library(janitor)

Warning message:
"Paket 'ggplot2' wurde unter R Version 4.4.3 erstellt"
Warning message:
"Paket 'purrr' wurde unter R Version 4.4.3 erstellt"
Warning message:
"Paket 'forcats' wurde unter R Version 4.4.1 erstellt"
-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.1     v stringr   1.5.1
v ggplot2   4.0.2     v tibble    3.2.1
v lubridate 1.9.3     v tidyr     1.3.1
v purrr     1.2.1     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x dplyr::filter() masks stats::filter()
x dplyr::lag()    masks stats::lag()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
"Paket 'glmnet' wurde unter R Version 4.4.1 erstellt"
Lade n"otiges Paket: Matrix


Attache Paket: 'Matrix'


Die folgenden Objekte sind maskiert von 'package:tidyr':

    expand, pack, unpack


Loaded glmnet 4.1-10

Warning message:
"Paket 'glm

In [21]:
df <- read_csv("data/merged_with_svi.csv") |> clean_names()
df <- df |> filter(!is.na(enrollment), enrollment > 0)
df <- df[, colSums(is.na(df)) == 0]
set.seed(100)
perc_strata = 0.75
strata <- ifelse(df$outbreak > 0, "nonzero", "zero")
index <- createDataPartition(strata, p = perc_strata, list = FALSE)
train <- df[index,]
test <- df[-index,]
outcome <- train$outbreak
offset <- log(train$enrollment)
phr <- train$phr

# train <- train %>% select(-county, -outbreak, -enrollment, -phr)


Rows: 254 Columns: 556
-- Column specification --------------------------------------------------------
Delimiter: ","
chr   (1): County
dbl (532): cve, outbreak, enrollment, population, PHR, pct_hispanic, pct_bla...
lgl  (23): median_income, Advised to Cut Down Salt - Do not use salt, Diabet...

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [4]:
X <- as.matrix(train)
y <- outcome
X <- X[, abs(cor(X, log(y + 1), use = "complete.obs", method = "spearman")) >= 0.1]

# Start with full list of candidates
candidates <- colnames(X)

i <- 1
while (i < length(candidates)) {
  j <- i + 1
  while (j <= length(candidates)) {
    rho <- cor(X[, candidates[i]], X[, candidates[j]], 
               method = "spearman", use = "complete.obs")
    if (abs(rho) >= 0.7) {
      # Drop whichever has weaker correlation with outcome
      rho_i <- abs(cor(X[, candidates[i]], log(y + 1), method = "spearman", use = "complete.obs"))
      rho_j <- abs(cor(X[, candidates[j]], log(y + 1), method = "spearman", use = "complete.obs"))
      if (rho_i >= rho_j) {
        candidates <- candidates[-j]  
      } else {
        candidates <- candidates[-i]  
        j <- i + 1
      }
    } else {
      j <- j + 1
    }
  }
  i <- i + 1
}

X <- X[, candidates]
X

cve,pct_hispanic,pct_uninsured,bmi_3_categories_recommended_range,clnscpy_sgmscpy_colonoscopy,e_cig_ever_not_at_all_right_now,ever_had_hiv_test_no,heavy_drinking_no,hysterectomy_no,kidney_disease_no,...,f_age65,f_age17,f_disabl,f_limeng,f_munit,e_nhpi,mp_afam,ep_aian,mp_aian,mp_twomore
2.54,19.6,18.5,25.4,83.3,21.3,62.4,94.9,71.4,94.6,...,0,0,0,0,0,33,0.9,0.7,0.5,1.0
2.50,23.5,17.7,23.8,75.5,20.6,55.7,93.1,64.6,95.9,...,0,0,0,0,0,0,0.9,0.4,0.4,0.9
5.24,12.0,4.2,21.7,87.6,18.5,63.5,95.9,81.1,96.5,...,1,0,0,0,0,0,1.7,1.1,1.0,1.0
1.08,65.4,19.5,28.7,84.6,18.4,59.1,91.0,80.0,95.9,...,0,0,0,0,0,0,0.3,0.0,0.1,0.5
0.80,66.0,28.6,21.7,87.6,18.5,63.5,95.9,81.1,96.5,...,0,0,0,1,0,0,0.4,1.0,0.9,1.4
3.78,21.0,13.3,28.7,84.6,18.4,59.1,91.0,80.0,95.9,...,1,0,0,0,0,0,0.4,0.5,0.4,1.1
2.09,45.0,21.8,34.1,86.3,19.3,60.4,92.8,83.0,95.7,...,0,0,0,0,0,0,0.5,0.1,0.1,0.9
1.48,61.8,18.8,27.0,77.9,22.3,60.4,97.3,74.2,93.0,...,0,0,0,0,0,0,0.7,0.3,0.3,0.7
2.42,26.3,14.0,34.1,86.3,19.3,60.4,92.8,83.0,95.7,...,0,0,0,0,0,2410,0.3,0.2,0.1,0.4
2.01,59.7,16.0,28.7,84.6,18.4,59.1,91.0,80.0,95.9,...,0,0,0,0,1,1995,0.1,0.1,0.1,0.1


In [8]:
library(bamlss)

fam <- beta_bamlss()
print(fam$names)

enrollment <- exp(offset)
rate <- y / enrollment

n <- length(rate)
rate_beta <- (rate * (n - 1) + 0.5) / n

dat <- data.frame(
  outbreak_rate = rate_beta,
  enrollment = enrollment,
  PHR = phr,
  X
)

xvars <- colnames(X)

la_terms <- paste0("la(", xvars, ")", collapse = " + ")

f <- as.formula(
  paste("outbreak_rate ~", la_terms)
)

fit <- bamlss(
  formula = f,
  family = fam,
  data = dat,
  sampler = FALSE,
  optimizer = opt_lasso,
  criterion = "BIC",
  nlambda = 100
)

[1] "mu"     "sigma2"


Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting 

BIC -1727.32 edf 40.494 lambda 1000.0 iteration   1

Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting 


BIC -1729.52 edf 40.784 lambda 869.74 iteration   2

...


BIC -1733.14 edf 47.999 lambda 0.0013 iteration  98

Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting 


BIC -1733.17 edf 47.999 lambda 0.0011 iteration  99

Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting NULL pointer to R NULL"
Warning message in .Call("xbin_fun", as.integer(ind), as.numeric(weights), as.numeric(e), :
"converting 


BIC -1733.20 edf 48.000 lambda 0.0010 iteration 100
elapsed time:  1.19min


In [10]:
mstop <- lasso_stop(fit)

coef_fit <- coef(fit, mstop = mstop)
print(coef_fit)

                                                                              mu.s.la(cve).cve 
                                                                                  1.742580e-01 
                                                                            mu.s.la(cve).tau21 
                                                                                  4.037017e-03 
                                                            mu.s.la(pct_hispanic).pct_hispanic 
                                                                                 -3.144989e-03 
                                                                   mu.s.la(pct_hispanic).tau21 
                                                                                  4.037017e-03 
                                                          mu.s.la(pct_uninsured).pct_uninsured 
                                                                                  2.940563e-02 
                                        

In [12]:
b <- unlist(coef_fit)

# keep only actual covariate coefficients from la() terms
selected <- b[
  grepl("^mu\\.s\\.la\\(", names(b)) &
    !grepl("\\.tau", names(b)) &
    abs(b) > 1e-5
]

selected_vars <- names(selected)

# extract variable names inside la(...)
selected_vars <- sub("^mu\\.s\\.la\\(([^\\)]+)\\).*", "\\1", selected_vars)

selected_table <- data.frame(
  variable = selected_vars,
  coefficient = as.numeric(selected),
  row.names = NULL
)

selected_table

selected_table <- selected_table[
  order(abs(selected_table$coefficient), decreasing = TRUE),
]

selected_table

variable,coefficient
<chr>,<dbl>
cve,1.742580e-01
pct_hispanic,-3.144989e-03
pct_uninsured,2.940563e-02
bmi_3_categories_recommended_range,-1.047090e-01
clnscpy_sgmscpy_colonoscopy,-4.069422e-02
e_cig_ever_not_at_all_right_now,-4.147698e-02
ever_had_hiv_test_no,6.047648e-04
heavy_drinking_no,7.785909e-04
hysterectomy_no,1.498899e-02


,variable,coefficient
,<chr>,<dbl>
1,cve,1.742580e-01
4,bmi_3_categories_recommended_range,-1.047090e-01
11,last_dentist_visit_within_the_past_2_years,-9.056295e-02
44,ep_aian,8.758621e-02
35,f_nohsdp,7.442897e-02
29,mp_age17,-6.088144e-02
40,f_limeng,-5.910460e-02
34,spl_themes,5.715761e-02
28,ep_age17,4.249937e-02


In [13]:
selected_vars <- selected_table$variable[
  abs(selected_table$coefficient) >= 0.03
]

In [23]:
library(betareg)
train$outbreak_rate <- train$outbreak / train$enrollment
test$outbreak_rate  <- test$outbreak / test$enrollment
n_train <- nrow(train)
n_test  <- nrow(test)

train$outbreak_rate <- (train$outbreak_rate * (n_train - 1) + 0.5) / n_train
test$outbreak_rate  <- (test$outbreak_rate  * (n_test - 1) + 0.5) / n_test

train_df_small <- train[, c("outbreak_rate", selected_vars), drop = FALSE]
test_df_small  <- test[,  c("outbreak_rate", selected_vars), drop = FALSE]

form_beta <- as.formula(
  paste("outbreak_rate ~", paste(selected_vars, collapse = " + "))
)

beta_fit <- betareg(
  form_beta,
  data = train_df_small
)

summary(beta_fit)


Call:
betareg(formula = form_beta, data = train_df_small)

Quantile residuals:
    Min      1Q  Median      3Q     Max 
-2.1334 -0.5121 -0.0153  0.5153  8.2095 

Coefficients (mean model with logit link):
                                             Estimate Std. Error z value
(Intercept)                                -10.561158   1.722634  -6.131
cve                                          0.197437   0.010952  18.028
bmi_3_categories_recommended_range          -0.014831   0.012074  -1.228
last_dentist_visit_within_the_past_2_years  -0.008514   0.014546  -0.585
ep_aian                                      0.255004   0.058650   4.348
f_nohsdp                                     0.593547   0.101685   5.837
mp_age17                                    -0.080714   0.023064  -3.500
f_limeng                                    -0.263309   0.121311  -2.171
spl_themes                                   0.106189   0.017494   6.070
ep_age17                                     0.046789   0.010757

In [24]:
selected_vars2 <- c(
  "cve",
  "ep_aian",
  "f_nohsdp",
  "mp_age17",
  "f_limeng",
  "spl_themes",
  "ep_age17",
  "clnscpy_sgmscpy_colonoscopy",
  "mp_twomore"
)

train_df_small2 <- train[, c("outbreak_rate", selected_vars2), drop = FALSE]
test_df_small2  <- test[,  c("outbreak_rate", selected_vars2), drop = FALSE]

form_beta2 <- as.formula(
  paste("outbreak_rate ~", paste(selected_vars2, collapse = " + "))
)

beta_fit2 <- betareg(
  form_beta2,
  data = train_df_small2
)

summary(beta_fit2)


Call:
betareg(formula = form_beta2, data = train_df_small2)

Quantile residuals:
    Min      1Q  Median      3Q     Max 
-2.0279 -0.4838 -0.0639  0.5214  8.1259 

Coefficients (mean model with logit link):
                              Estimate Std. Error z value Pr(>|z|)    
(Intercept)                 -10.507868   0.799033 -13.151  < 2e-16 ***
cve                           0.194595   0.010302  18.889  < 2e-16 ***
ep_aian                       0.240619   0.056217   4.280 1.87e-05 ***
f_nohsdp                      0.587833   0.100328   5.859 4.65e-09 ***
mp_age17                     -0.074936   0.022094  -3.392 0.000695 ***
f_limeng                     -0.313338   0.118571  -2.643 0.008227 ** 
spl_themes                    0.102660   0.017446   5.884 3.99e-09 ***
ep_age17                      0.047258   0.007389   6.395 1.60e-10 ***
clnscpy_sgmscpy_colonoscopy   0.025212   0.008891   2.836 0.004574 ** 
mp_twomore                    0.034996   0.021367   1.638 0.101463    

Phi coeffi

In [25]:
pred_test <- predict(beta_fit2, newdata = test_df_small2, type = "response")

test_y <- test_df_small2$outbreak_rate

rmse <- sqrt(mean((test_y - pred_test)^2))
mae  <- mean(abs(test_y - pred_test))

sse <- sum((test_y - pred_test)^2)
sst <- sum((test_y - mean(test_y))^2)
r2_test <- 1 - sse / sst

c(RMSE = rmse, MAE = mae, R2_test = r2_test)

RMSE          MAE      R2_test 
 0.006200048  0.005703789 -4.605358968

In [26]:
pred_test1 <- predict(beta_fit,  newdata = test_df_small,  type = "response")
pred_test2 <- predict(beta_fit2, newdata = test_df_small2, type = "response")

eval <- function(y, pred) {
  c(
    RMSE = sqrt(mean((y - pred)^2)),
    MAE = mean(abs(y - pred)),
    R2_test = 1 - sum((y - pred)^2) / sum((y - mean(y))^2)
  )
}

rbind(
  model_14_vars = eval(test_df_small$outbreak_rate, pred_test1),
  model_9_vars  = eval(test_df_small2$outbreak_rate, pred_test2)
)

,RMSE,MAE,R2_test
model_14_vars,0.006181510,0.005719509,-4.571889
model_9_vars,0.006200048,0.005703789,-4.605359


In [27]:
test_y <- test_df_small2$outbreak_rate

baseline_pred <- rep(mean(test_y), length(test_y))

baseline_rmse <- sqrt(mean((test_y - baseline_pred)^2))
baseline_mae  <- mean(abs(test_y - baseline_pred))

c(
  model_RMSE = rmse,
  baseline_RMSE = baseline_rmse,
  model_MAE = mae,
  baseline_MAE = baseline_mae
)

model_RMSE baseline_RMSE     model_MAE  baseline_MAE 
 0.0062000476  0.0026187456  0.0057037887  0.0007889461